# 02 — EDA (Data Understanding)

**Fase 2 — CRISP-DM: Data Understanding** · Cubre las Tareas 1–5 del enunciado (Criterios C1 + C2, 50% de la nota).

Este notebook trabaja sobre el dataset **crudo, sin modificar** (`data/raw/act_liver_disease.csv`, inmutable). No se imputa, no se elimina ningún registro y no se entrena ningún modelo — el objetivo es entender estructura, calidad, distribuciones y relaciones entre variables. Las decisiones de tratamiento (imputación, escalado, outliers) se toman en `03_preprocessing.ipynb`.

Ver `docs/adr/0003-sin-holdout-en-fases-0-3.md` para la decisión de no apartar ningún *holdout* antes de este análisis.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from src.config import CORR_THRESHOLD, DPI, NUMERIC_COLS
from src.utils import load_raw_data, save_figure

sns.set_theme(style="whitegrid")

df = load_raw_data()
df.shape

Matplotlib is building the font cache; this may take a moment.


(583, 11)

## T1 — Estructura del dataset

¿Cuál es la estructura del dataset? Formato, tamaño, cantidad de variables y tipo de datos.

In [2]:
print("Formato: CSV delimitado por comas")
print("Shape (filas, columnas):", df.shape)
print("\nTipos de datos:")
print(df.dtypes)
print(f"\nMemoria en uso: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
df.head(3)

Formato: CSV delimitado por comas
Shape (filas, columnas): (583, 11)

Tipos de datos:
Age            int64
Gender           str
TB           float64
DB           float64
Alkphos        int64
Sgpt           int64
Sgot           int64
TP           float64
ALB          float64
A/G Ratio    float64
Selector       int64
dtype: object

Memoria en uso: 76.1 KB


,Age,Gender,TB,DB,Alkphos,Sgpt,Sgot,TP,ALB,A/G Ratio,Selector
0,65,Female,0.7,0.1,187,16,18,6.8,3.3,0.90,1
1,62,Male,10.9,5.5,699,64,100,7.5,3.2,0.74,1
2,62,Male,7.3,4.1,490,60,68,7.0,3.3,0.89,1


**Interpretación.** El dataset tiene **583 filas × 11 columnas** (≈76 KB en memoria) en formato CSV delimitado por comas. De las 11 columnas: **9 son numéricas** (`Age`, `TB`, `DB`, `Alkphos`, `Sgpt`, `Sgot`, `TP`, `ALB`, `A/G Ratio` — mezcla de `int64` y `float64`), **1 es categórica** (`Gender`, texto Male/Female) y **1 es el target** (`Selector`, `int64`, codificado 1/2). No hay columnas de identificador ni de fecha.

Nota sobre los tipos: `Alkphos`, `Sgpt` y `Sgot` son enteros porque los equipos de laboratorio que las miden (fosfatasa alcalina, ALT, AST) reportan unidades enteras (IU/L); `TB`, `DB`, `TP`, `ALB` y `A/G Ratio` son decimales por la precisión de la medición bioquímica.

## T2 — Problemas de calidad

¿Qué problemas de calidad existen en el dataset, como datos faltantes o errores?

In [3]:
print("--- Q2: valores faltantes por columna ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n--- Q3: duplicados exactos ---")
print("Filas duplicadas:", df.duplicated().sum())
print("Filas unicas:", df.shape[0] - df.duplicated().sum())

print("\n--- Q4: desbalance de clase (Selector) ---")
print(df["Selector"].value_counts())
print((df["Selector"].value_counts(normalize=True) * 100).round(1))

print("\n--- Q5: desbalance de sexo (Gender) ---")
print(df["Gender"].value_counts())
print((df["Gender"].value_counts(normalize=True) * 100).round(1))

print("\n--- Q6: codificacion del target ---")
print("Valores unicos de Selector:", sorted(df["Selector"].unique()))

print("\n--- Q7: rango de edad ---")
print(df["Age"].describe())
print("Pacientes con Age == 90:", (df["Age"] == 90).sum())

--- Q2: valores faltantes por columna ---
A/G Ratio    4
dtype: int64

--- Q3: duplicados exactos ---
Filas duplicadas: 13
Filas unicas: 570

--- Q4: desbalance de clase (Selector) ---
Selector
1    416
2    167
Name: count, dtype: int64
Selector
1    71.4
2    28.6
Name: proportion, dtype: float64

--- Q5: desbalance de sexo (Gender) ---
Gender
Male      441
Female    142
Name: count, dtype: int64
Gender
Male      75.6
Female    24.4
Name: proportion, dtype: float64

--- Q6: codificacion del target ---
Valores unicos de Selector: [np.int64(1), np.int64(2)]

--- Q7: rango de edad ---
count    583.000000
mean      44.746141
std       16.189833
min        4.000000
25%       33.000000
50%       45.000000
75%       58.000000
max       90.000000
Name: Age, dtype: float64
Pacientes con Age == 90: 1


**Interpretación — 7 problemas de calidad detectados:**

1. **Valores faltantes:** solo `A/G Ratio` tiene nulos, **4 de 583** (0.7%). Es la única columna derivada algebraicamente (`ALB / (TP − ALB)`), lo que sugiere que los 4 faltantes ocurrieron en el cálculo original, no en la medición de `TP`/`ALB`. Justifica la Tarea 6 (imputación).
2. **Filas duplicadas exactas:** **13 duplicados** (570 filas únicas de 583). El enunciado pregunta explícitamente por "errores" en T2, así que esto se trata como un hallazgo obligatorio, no un extra — la decisión sobre qué hacer con ellos (¿pacientes distintos con analítica idéntica, o error de captura?) se documenta en la Fase 3 (T2/F3-R13), no aquí, para no alterar todavía el dataset de referencia de este notebook.
3. **Desbalance de clase:** **416 pacientes clase 1 (enfermo) vs. 167 clase 2 (sano)**, ≈71.4% / 28.6%. La *accuracy* será una métrica engañosa en el modelado futuro (un clasificador que dijera "todos enfermos" acertaría 71% sin aprender nada).
4. **Desbalance de sexo:** **441 hombres vs. 142 mujeres**, ≈75.6% / 24.4%. Esta es la raíz técnica del sesgo documentado por Straw & Wu (2022) — con tan pocas mujeres en la muestra, cualquier modelo tiene menos señal para aprender patrones específicos de ese subgrupo.
5. **Codificación contraintuitiva del target:** `Selector` toma valores **{1, 2}**, donde **1 = enfermo** y 2 = sano — no es el 0/1 estándar y el orden no es el intuitivo ("1" no significa "positivo/no enfermo"). Deberá recodificarse antes de cualquier modelado.
6. **Rango de edad:** de **4 a 90 años**, con un único paciente en el extremo superior (Age = 90) y 14 pacientes en Age = 75. No hay evidencia de que 90 sea un *cap* de captura (un valor artificial tipo "90+"): es un solo registro, sin acumulación anómala en ese valor. Se reporta como verificado, no concluyente — no se afirma la existencia de censura de edad.
7. **Sin errores de formato:** no se encontraron valores no numéricos en columnas numéricas, ni categorías inesperadas en `Gender` (solo Male/Female), ni columnas con tipo de dato inconsistente.

Ningún otro problema de calidad estructural (columnas vacías, filas totalmente nulas, *encoding* corrupto) fue detectado.

## T3 — Estadística descriptiva de `TB` y `DB`

¿Cuáles son la media, la desviación estándar, la varianza, el rango y el rango intercuartílico de `TB` y `DB`? ¿Qué implicaciones tienen para la variabilidad de los datos?

> Se calcula sobre los datos **originales, sin imputar** — ninguna de las dos columnas tiene faltantes, así que no hay diferencia con la versión de la Fase 3, pero se deja explícito por consistencia con la regla del PRD de no mezclar estadísticos con datos ya tratados.

In [4]:
rows = []
for col in ["TB", "DB"]:
    s = df[col]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    rows.append({
        "variable": col,
        "media": s.mean(),
        "mediana": s.median(),
        "std": s.std(),
        "varianza": s.var(),
        "min": s.min(),
        "max": s.max(),
        "rango": s.max() - s.min(),
        "IQR": q3 - q1,
    })
t3_summary = pd.DataFrame(rows).set_index("variable").round(3)
t3_summary

,media,mediana,std,varianza,min,max,rango,IQR
variable,,,,,,,,
TB,3.299,1.0,6.210,38.558,0.4,75.0,74.6,1.8
DB,1.486,0.3,2.808,7.888,0.1,19.7,19.6,1.1


**Interpretación.**

| Variable | Media | Mediana | Std | Varianza | Rango | IQR |
|---|---|---|---|---|---|---|
| `TB` | 3.30 | 1.00 | 6.21 | 38.56 | 74.60 (0.4–75.0) | 1.80 |
| `DB` | 1.49 | 0.30 | 2.81 | 7.89 | 19.60 (0.1–19.7) | 1.10 |

**Media vs. mediana — evidencia de asimetría fuerte:** en ambas variables la media es **muy superior** a la mediana (`TB`: 3.30 vs. 1.00; `DB`: 1.49 vs. 0.30) — la media casi triplica/cuadruplica a la mediana. Esto es la firma numérica de una distribución con **cola derecha larga**: unos pocos pacientes con bilirrubina muy alta arrastran la media hacia arriba, mientras que la mayoría de los pacientes tiene valores bajos, cercanos a la mediana. Se confirma con *skewness*/*kurtosis* en T4.

**Rango total vs. IQR — cuánto "estiran" los extremos:** el rango de `TB` (74.6) es **~41 veces** su IQR (1.8); el de `DB` (19.6) es **~18 veces** su IQR (1.1). El IQR —robusto a extremos— describe la dispersión del 50% central de los pacientes, y es mucho más pequeño que el rango total. La brecha enorme entre ambos indica que un número reducido de pacientes muy graves (bilirrubina extremadamente alta) está "estirando" la distribución muy por encima de donde vive la mayoría de los datos. Esta no es una anomalía a corregir: es la **señal clínica** que un modelo de cribado necesita detectar — los outliers de bilirrubina son, potencialmente, los pacientes que más urge derivar al especialista (se retoma en T8).

## T4 — Forma y simetría de `TB`, `DB`, `Alkphos`, `Sgpt`

¿Qué forma y simetría tienen las distribuciones? Histogramas y análisis de los patrones observados.

In [5]:
t4_cols = ["TB", "DB", "Alkphos", "Sgpt"]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col in zip(axes.flat, t4_cols):
    sns.histplot(df[col], bins=30, kde=True, ax=ax, color="#4C72B0")
    ax.set_title(col)
    ax.set_xlabel(col)
fig.suptitle("T4 — Distribuciones de biomarcadores (dataset completo, n=583)")
fig.tight_layout()
save_figure(fig, "t4_hist_bilirubin_enzymes.png")
plt.show()

C:\Users\harrison.tutalcha\AppData\Local\Temp\ipykernel_32096\2098913191.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
skew_kurt = pd.DataFrame({
    "skewness": df[NUMERIC_COLS].skew(),
    "kurtosis_fisher": df[NUMERIC_COLS].kurtosis(),
}).round(3)
skew_kurt

,skewness,kurtosis_fisher
Age,-0.029,-0.560
TB,4.907,37.164
DB,3.212,11.353
Alkphos,3.765,17.753
Sgpt,6.549,50.579
Sgot,10.546,150.920
TP,-0.286,0.233
ALB,-0.044,-0.388
A/G Ratio,0.992,3.282


**Interpretación.** Los cuatro histogramas muestran el mismo patrón: una barra alta cerca de cero y una cola larga hacia la derecha, con muy pocos casos aislados en valores extremos. La *skewness* lo confirma numéricamente:

- `TB`: skew = **4.91**, kurtosis = **37.2**
- `DB`: skew = **3.21**, kurtosis = **11.4**
- `Alkphos`: skew = **3.77**, kurtosis = **17.8**
- `Sgpt`: skew = **6.55**, kurtosis = **50.6**

Todas tienen **skewness fuertemente positivo** (>0, cola derecha) y **kurtosis muy por encima de 0** (colas mucho más pesadas que una normal — la referencia de "sin exceso" es kurtosis de Fisher = 0). `Sgot` (no graficada aquí, se retoma en T8) es la más extrema de las nueve variables numéricas: skew = 10.55, kurtosis = 150.9.

**Lectura clínica, no estadística:** esta asimetría **no es ruido a corregir** — es exactamente lo esperado en biomarcadores de daño hepático. La mayoría de los pacientes de este dataset tiene función hepática cercana a la normalidad (por eso la masa de datos se concentra cerca de valores bajos), y una minoría con daño severo produce los valores extremos que alargan la cola. Para contraste, `Age`, `TP` y `ALB` tienen skew cercano a 0 (-0.03, -0.29, -0.04 respectivamente) — son las variables demográficas/de síntesis que sí siguen una forma aproximadamente simétrica. `A/G Ratio`, al ser un cociente de dos variables casi simétricas, tiene una asimetría moderada (skew = 0.99), notablemente menor que las enzimas y bilirrubinas.

## T5 — Matriz de correlación

¿Qué relaciones existen entre las variables? Matriz de correlación y correlaciones significativas que puedan influir en la selección de variables.

> Se calculan **Pearson** (relación lineal) y **Spearman** (relación monótona, robusta a la fuerte asimetría de T4) — con distribuciones tan sesgadas, Spearman es la referencia más confiable.

In [7]:
corr_pearson = df[NUMERIC_COLS].corr(method="pearson")
corr_spearman = df[NUMERIC_COLS].corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(corr_pearson, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Pearson")
sns.heatmap(corr_spearman, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("Spearman")
fig.suptitle("T5 — Matriz de correlación (variables numéricas, n=583)")
fig.tight_layout()
save_figure(fig, "t5_heatmap_correlacion.png")
plt.show()

C:\Users\harrison.tutalcha\AppData\Local\Temp\ipykernel_32096\1732473117.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
pairs = []
for i in range(len(NUMERIC_COLS)):
    for j in range(i + 1, len(NUMERIC_COLS)):
        r_p = corr_pearson.iloc[i, j]
        if abs(r_p) > CORR_THRESHOLD:
            pairs.append({
                "var_1": NUMERIC_COLS[i],
                "var_2": NUMERIC_COLS[j],
                "pearson": round(r_p, 3),
                "spearman": round(corr_spearman.iloc[i, j], 3),
            })
pares_altos = pd.DataFrame(pairs)
pares_altos

,var_1,var_2,pearson,spearman
0,TB,DB,0.875,0.959
1,Sgpt,Sgot,0.792,0.774
2,TP,ALB,0.784,0.779


**Interpretación.** Tres pares superan el umbral `CORR_THRESHOLD = 0.7`:

| Grupo funcional | Par | Pearson | Spearman |
|---|---|---|---|
| Excreción | `TB` ↔ `DB` | **0.875** | **0.959** |
| Daño celular | `Sgpt` ↔ `Sgot` | **0.792** | **0.774** |
| Síntesis | `TP` ↔ `ALB` | **0.784** | **0.779** |

(Referencia adicional, justo debajo del umbral: `ALB` ↔ `A/G Ratio`, Pearson 0.690 / Spearman 0.754 — relevante porque conecta con la dependencia algebraica de la Sección 5.2 del PRD.)

**Conexión con la selección de variables — decisión por grupo:**

| Grupo | Variable a conservar | Razón |
|---|---|---|
| `TB` ↔ `DB` | **`TB`** | `DB` está *contenida* en `TB` (bilirrubina directa es una fracción de la total) — es redundancia **estructural**, no casual. Spearman = 0.96 indica que `TB` por sí sola retiene casi toda la señal ordinal de `DB`. `TB` es además el marcador de cribado inicial más usado clínicamente. |
| `Sgpt` ↔ `Sgot` | **Ambas, con reserva** | La correlación es alta pero no estructural (son enzimas de órganos distintos: ALT/`Sgpt` es hepato-específica, AST/`Sgot` también existe en corazón y músculo). Straw & Wu (2022) encuentran que **pesan distinto según el sexo** — eliminar una de entrada podría borrar señal relevante justo para el análisis de *fairness* que motiva este proyecto. Se recomienda evaluar el *Variance Inflation Factor* (VIF) en la Fase 4 antes de descartar alguna. |
| `TP` ↔ `ALB` (+ `A/G Ratio`) | **`ALB` y `A/G Ratio`** | `TP = ALB + Globulina`, dependencia estructural parcial. `ALB` es el biomarcador de síntesis hepática más directo y clínicamente interpretable (baja con la enfermedad); `A/G Ratio` ya combina algebraicamente la información de `TP` y `ALB` en una sola variable derivada. `TP` es, de las tres, la más redundante. |

**Lectura general:** correlación alta entre predictores significa **redundancia de información**, no necesariamente causalidad. Para el modelado futuro (Fase 4) convendrá partir de un subconjunto que evite duplicar señal — pero la decisión final de qué excluir debe hacerse *dentro* de esa fase, con herramientas cuantitativas (VIF, importancia de variables) y no solo con el umbral de correlación, precisamente por el caso `Sgpt`/`Sgot` señalado arriba.

## Valor agregado — Análisis estratificado por `Gender`

El enunciado no lo exige, pero es el eje diferenciador de este proyecto (§0.2 del PRD). Se hace **después** del análisis agregado, nunca en su reemplazo.

In [9]:
print("--- Medias por sexo (variables numéricas) ---")
display(df.groupby("Gender")[NUMERIC_COLS].mean().round(3))

print("\n--- Tabla de contingencia Gender x Selector ---")
display(pd.crosstab(df["Gender"], df["Selector"]))

print("\n--- Tasa de diagnóstico positivo (Selector==1) por sexo ---")
tasa_positivo = df.groupby("Gender")["Selector"].apply(lambda s: (s == 1).mean() * 100).round(2)
display(tasa_positivo)

--- Medias por sexo (variables numéricas) ---


,Age,TB,DB,Alkphos,Sgpt,Sgot,TP,ALB,A/G Ratio
Gender,,,,,,,,,
Female,43.134,2.323,0.989,302.338,54.239,69.042,6.654,3.273,0.949
Male,45.265,3.613,1.646,286.789,89.238,123.070,6.428,3.100,0.946



--- Tabla de contingencia Gender x Selector ---


Selector,1,2
Gender,,
Female,92,50
Male,324,117



--- Tasa de diagnóstico positivo (Selector==1) por sexo ---


Gender
Female    64.79
Male      73.47
Name: Selector, dtype: float64

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col in zip(axes, ["TB", "Alkphos"]):
    sns.histplot(data=df, x=col, hue="Gender", bins=30, kde=True, ax=ax, palette=["#4C72B0", "#DD8452"], element="step")
    ax.set_title(f"{col} por sexo")
fig.suptitle("Distribuciones estratificadas por Gender")
fig.tight_layout()
save_figure(fig, "eda_extra_hist_por_sexo.png")
plt.show()

C:\Users\harrison.tutalcha\AppData\Local\Temp\ipykernel_32096\4068109723.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretación.** Dos hallazgos relevantes para la pregunta de investigación 2 (¿hay diferencias de biomarcadores por sexo?):

1. **Los hombres tienen enzimas y bilirrubinas más altas en promedio:** `TB` (3.61 vs. 2.32), `DB` (1.65 vs. 0.99), `Sgpt` (89.2 vs. 54.2), `Sgot` (123.1 vs. 69.0) — todas más altas en hombres que en mujeres. Es consistente con que la muestra masculina tiene proporcionalmente más casos graves (ver tasa de diagnóstico abajo).
2. **`Alkphos` (ALP) rompe el patrón:** es la **única** enzima con media más alta en mujeres (302.3) que en hombres (286.8), pese a que las mujeres tienen valores más bajos en todo lo demás. Esto coincide con lo que reporta Straw & Wu (2022): al corregir la subrepresentación femenina, **ALP y sexo se vuelven las variables más importantes** del modelo — este dataset ya muestra la semilla de ese patrón a nivel descriptivo.

**Tasa de diagnóstico positivo por sexo — hallazgo que conecta con el *label bias* (Sección 2, Fase 1):** el 73.47% de los hombres tiene `Selector=1` (enfermo) frente al **64.79% de las mujeres** (tabla de contingencia: 324/117 hombres, 92/50 mujeres). A partir de biomarcadores más bajos en promedio, es razonable que la tasa de diagnóstico también sea menor en mujeres — pero **no podemos distinguir, solo con este dataset, si esa diferencia refleja una población realmente más sana o un patrón de sub-diagnóstico** (el segundo es justamente lo que reporta la literatura citada en la Fase 1 sobre menor sospecha clínica en mujeres). Se deja como pregunta abierta explícita para la auditoría de *fairness* de la Fase 5 (fuera de este PRD), no se puede responder con EDA.

> 🔁 **Nota de iteración (Loop A):** este hallazgo (diferencia de ~9 puntos porcentuales en tasa de diagnóstico por sexo, más el patrón atípico de `Alkphos`) es justo el tipo de resultado que la Fase 1 anticipaba que podía reforzar la pregunta de investigación 3. Se registra en `docs/CHANGELOG_iteraciones.md`.

## Valor agregado — Verificación de la fórmula `A/G Ratio = ALB / (TP − ALB)`

Prepara la decisión de imputación determinista de la Fase 3 (F3-R5 del PRD): si la fórmula reconstruye `A/G Ratio` con error bajo, es una alternativa superior a imputar por media.

In [11]:
print("Filas con TP == ALB (riesgo de división por cero):", (df["TP"] == df["ALB"]).sum())

comp = df.dropna(subset=["A/G Ratio", "TP", "ALB"]).copy()
comp["AG_calc"] = comp["ALB"] / (comp["TP"] - comp["ALB"])
comp["error_abs"] = (comp["AG_calc"] - comp["A/G Ratio"]).abs()

print(f"\nFilas comparables (sin nulos): {len(comp)}")
print("\nError absoluto de reconstrucción:")
print(comp["error_abs"].describe().round(4))
print("\nFilas con error > 0.05:", (comp["error_abs"] > 0.05).sum(), f"({(comp['error_abs'] > 0.05).mean()*100:.1f}%)")

Filas con TP == ALB (riesgo de división por cero): 0

Filas comparables (sin nulos): 579

Error absoluto de reconstrucción:
count    579.0000
mean       0.0514
std        0.1664
min        0.0000
25%        0.0067
50%        0.0310
75%        0.0615
max        2.3000
Name: error_abs, dtype: float64

Filas con error > 0.05: 191 (33.0%)


**Interpretación — hallazgo más matizado de lo esperado.** No hay riesgo de división por cero (0 filas con `TP == ALB`). Sobre las 579 filas comparables, la reconstrucción algebraica **no es perfectamente exacta**: error absoluto medio = **0.051**, mediana = **0.031**, pero con una cola de casos peores (máximo 2.30) y **191 filas (33.0%) con error > 0.05**.

Esto no invalida la fórmula — la relación `A/G Ratio = ALB / (TP − ALB)` es correcta por definición bioquímica (globulina = proteína total − albúmina) — pero indica que **los valores originales del dataset están redondeados** antes de guardarse (`TP` y `ALB` a un decimal, `A/G Ratio` a dos), y dividir dos cantidades ya redondeadas amplifica el error relativo, especialmente cuando `TP − ALB` (la globulina) es pequeña. Es un matiz importante para la Fase 3: la imputación determinista sigue siendo **conceptualmente superior** a la media (no depende de la distribución de otros pacientes), pero no es "exacta" en la práctica sobre este dataset concreto — se reportará así, con honestidad, en vez de asumir un error cercano a cero.

## Notas para fases futuras (no se ejecuta nada aquí)

- **Fase 3 (Preprocessing):** usar el hallazgo de la sección anterior para comparar honestamente imputación por media vs. determinista — el error de reconstrucción de 0.05 en promedio (33% de filas por encima de ese umbral) debe citarse tal cual, no redondearse a "la fórmula es exacta".
- **Fase 3 (T2/duplicados):** decidir el tratamiento de los 13 duplicados exactos detectados aquí; no se eliminan en este notebook para no alterar los números reportados en T1–T5.
- **Fase 4+ (modelado, fuera de este PRD):** `Selector` debe recodificarse (1/2 → 1/0) antes de entrenar cualquier modelo. El par `Sgpt`/`Sgot` requiere evaluación de VIF antes de decidir si se descarta alguna. El split de train/test para modelado y auditoría de *fairness* debe ser sustancialmente mayor a 29 filas y estratificado por `Selector` y `Gender` (ver `docs/adr/0003-sin-holdout-en-fases-0-3.md`).